# 01 — Load the eight samples

Equivalent to `scripts/01_load_data.py`. Merges the 8 CellRanger outputs into one
AnnData and attaches sex / treatment / replicate to every cell.

**The critical part is the metadata.** If the labels are wrong here, everything
downstream is wrong in a way that never raises an error.

In [ ]:
import sys

from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import scanpy as sc

import anndata as ad



import config



sc.settings.verbosity = 3

sc.settings.figdir = config.FIG_DIR

sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

sc.logging.print_header()

## Find the sample folders

In [ ]:
sample_dirs = sorted(
    p for p in config.DATA_DIR.iterdir()
    if p.is_dir() and (p / 'matrix.mtx.gz').exists()
)
print(f'Found {len(sample_dirs)} samples:')
for p in sample_dirs:
    print(' ', p.name)

## Load each one, attaching metadata

`var_names='gene_symbols'` gives readable names (`repo`, `elav`, `ple`) rather than
FlyBase IDs. If you end up with `FBgn...` names, the mitochondrial detection in
step 02 silently finds nothing.

In [ ]:
adatas = {}

for path in sample_dirs:
    name = path.name
    sex, treatment, replicate = name.split('_')
    assert sex in {'Female', 'Male'}, f'bad sex in {name}'
    assert treatment in {'Cocaine', 'Sucrose'}, f'bad treatment in {name}'

    a = sc.read_10x_mtx(path, var_names='gene_symbols', cache=True)
    a.var_names_make_unique()

    a.obs['sample'] = name
    a.obs['sex'] = sex
    a.obs['treatment'] = treatment
    a.obs['replicate'] = replicate
    a.obs['condition'] = f'{sex}_{treatment}'

    print(f'{name}: {a.n_obs:,} cells x {a.n_vars:,} genes')
    adatas[name] = a

## Concatenate

`join='outer'` keeps every gene seen in any sample. `join='inner'` would silently
drop genes missing from a single sample — possibly a marker you need later.
`index_unique='-'` stops identical barcodes from different runs colliding.

In [ ]:
adata = ad.concat(adatas, label='sample_batch', index_unique='-', join='outer')
adata.var_names_make_unique()

for col in ['sample', 'sex', 'treatment', 'replicate', 'condition']:
    adata.obs[col] = adata.obs[col].astype('category')

print(f'Merged: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
adata

## Check the design survived

This should be a full 2×2×2. A missing cell in this table means a sample failed
to load.

In [ ]:
pd.crosstab(adata.obs['sex'], [adata.obs['treatment'], adata.obs['replicate']])

In [ ]:
adata.write(config.H5AD_RAW)
print('Wrote', config.H5AD_RAW)

---
**Next:** run `tools/verify_sample_mapping.py` before going any further.
It checks `roX1`/`roX2` to catch an inverted sex label. Then `02_qc_filter.ipynb`.